# Viscosity Pipeline Usage Example

This notebook shows how to run the reusable viscosity pipeline on:
results/auto_runs/dynamic_analysis_test_prediction_37kcP_custom_20260525_091234.csv

Workflow covered:
1. Load dataset
2. Normalize x-axis by cell
3. Trim before-hit and after-contact zones
4. Fit Polynomial(2nd) and Hyperbola
5. Map known real viscosities (user input)
6. Extrapolate unknown viscosities
7. Plot fits and accuracy

In [10]:
import sys
from pathlib import Path

def _find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "viscometry").is_dir():
            return p
    raise RuntimeError("Could not locate repo root (expected src/viscometry)")

PROJECT_ROOT = _find_repo_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

AUTO_RUNS = PROJECT_ROOT / "results" / "auto_runs"
AUTO_RUNS_LEGACY = PROJECT_ROOT / "results" / "Auto-runs"
ARCHIVE = PROJECT_ROOT / "results" / "runs" / "archive"

import pandas as pd
from viscometry.analysis.viscosity_pipeline import run_viscosity_pipeline

In [11]:
csv_name = "dynamic_analysis_test_prediction_37kcP_custom_20260525_091234.csv"
_csv_candidate = AUTO_RUNS / csv_name
if not _csv_candidate.is_file():
    _csv_candidate = ARCHIVE / csv_name
if not _csv_candidate.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_name}")
csv_path = str(_csv_candidate.resolve())

real_viscosity_map = {
    1: 37000,
}

In [12]:
pipeline_out = run_viscosity_pipeline(
    csv_path=csv_path,
    real_viscosity_map=real_viscosity_map,
    visualize=True,
)

print('Learned scaling factors:', pipeline_out['scales'])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 3748: invalid start byte

In [ ]:
pred_cols = [
    'cell',
    'a_poly2',
    'a_hyperbola',
    'real_viscosity',
    'predicted_visc_pol',
    'predicted_visc_hyp',
    'rel_error_pol',
    'rel_error_hyp',
    'is_calibration',
]

pred_df = pipeline_out['predictions'][pred_cols].sort_values('cell').reset_index(drop=True)
display(pred_df)

# Optional: save the predictions table
# pred_df.to_csv('viscosity_predictions_dynamic_analysis_L60kcP.csv', index=False)